# Recherche d'hyperparamètres — HOG, SIFT, ORB

**Stratégie données** : échantillon stratifié de 5 images/classe (230 images max) chargées en RAM une seule fois. Pour chaque combinaison de paramètres, seule l'extraction est refaite — pas le I/O disque.  
**Sklearn** : chaque descripteur est un `BaseEstimator`. `fit()` construit la base de features, `score()` calcule la MAP. `GridSearchCV` avec `StratifiedKFold(3)` orchestre le tout.

In [3]:
import os, sys, warnings, time
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from sklearn.base import BaseEstimator
from sklearn.model_selection import GridSearchCV, StratifiedKFold

warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath('.')
DATASET_DIR  = os.path.join(PROJECT_ROOT, 'dataset')
INDEX_DIR    = os.path.join(PROJECT_ROOT, 'indexes')

print('Dataset :', DATASET_DIR)
print('Indexes :', INDEX_DIR)

Dataset : c:\MIR_Project\dataset
Indexes : c:\MIR_Project\indexes


## 1. Chargement de l'échantillon stratifié

In [4]:
N_PER_CLASS = 5   # images par classe → 46 × 5 = 230 images max

filenames = np.load(os.path.join(INDEX_DIR, 'filenames.npy'))
classes   = np.load(os.path.join(INDEX_DIR, 'classes.npy'))

# Échantillonnage stratifié
selected_idx = []
for cls in np.unique(classes):
    idx = np.where(classes == cls)[0]
    chosen = idx[:N_PER_CLASS]          # premières N images de la classe
    selected_idx.extend(chosen.tolist())

selected_idx = np.array(selected_idx)
sel_filenames = filenames[selected_idx]
sel_classes   = classes[selected_idx]

# Encodage numérique des classes
unique_cls, y = np.unique(sel_classes, return_inverse=True)

print(f'{len(selected_idx)} images sélectionnées, {len(unique_cls)} classes')

230 images sélectionnées, 46 classes


In [5]:
# Chargement en RAM (une seule fois pour tout le notebook)
images_ram = []
valid_mask = []

for fname in tqdm(sel_filenames, desc='Chargement images'):
    path = os.path.join(DATASET_DIR, fname)
    img  = cv2.imread(path)
    if img is None:
        images_ram.append(np.zeros((128, 128, 3), dtype=np.uint8))
        valid_mask.append(False)
    else:
        images_ram.append(img)
        valid_mask.append(True)

images_ram = np.array(images_ram, dtype=object)   # tableau de BGR arrays
valid_mask = np.array(valid_mask)
print(f'{valid_mask.sum()} images valides sur {len(valid_mask)}')

ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

## 2. Fonctions utilitaires

In [ ]:
def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    """Normalisation L2 ligne par ligne."""
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return matrix / norms


def compute_map(features: np.ndarray, labels: np.ndarray, top_k: int = 50) -> float:
    """
    Calcule la MAP (Mean Average Precision) sur l'ensemble donné.
    Chaque image sert de requête contre les (N-1) autres.
    """
    feats = l2_normalize(features.astype(np.float32))
    N = len(feats)
    aps = []

    # Matrice de distances cosinus (= 1 - produit scalaire sur L2-normalisé)
    sim_matrix = feats @ feats.T          # (N, N) similarités

    for i in range(N):
        sims = sim_matrix[i].copy()
        sims[i] = -np.inf                 # exclure la requête elle-même
        ranked = np.argsort(-sims)        # tri décroissant de similarité

        relevant = (labels[ranked] == labels[i]).astype(int)
        n_relevant = relevant.sum()
        if n_relevant == 0:
            continue

        k = min(top_k, N - 1)
        rel_k = relevant[:k]
        cumsum = np.cumsum(rel_k)
        precision_at_k = cumsum / (np.arange(1, k + 1))
        ap = (precision_at_k * rel_k).sum() / n_relevant
        aps.append(ap)

    return float(np.mean(aps)) if aps else 0.0


print('Utilitaires chargés.')

## 3. Estimateurs sklearn

Chaque descripteur hérite de `BaseEstimator` :
- `fit(X, y)` : extrait les features de toutes les images `X` (liste de BGR arrays), stocke en `db_features_`
- `score(X, y)` : calcule la MAP pour les images `X` (requêtes) contre `db_features_`

In [ ]:
# ─── HOG Estimator ────────────────────────────────────────────────────────

class HOGRetriever(BaseEstimator):
    """
    Paramètres :
      win_size   : taille fenêtre (px), carré
      block_size : taille bloc (px), carré
      cell_size  : taille cellule (px), carré
      nbins      : nombre de bins d'orientation
    Note : block_stride = block_size // 2 (fixé à 50 % de recouvrement)
    """
    def __init__(self, win_size=128, block_size=16, cell_size=8, nbins=9):
        self.win_size   = win_size
        self.block_size = block_size
        self.cell_size  = cell_size
        self.nbins      = nbins

    def _build_hog(self):
        bs = self.block_size
        cs = self.cell_size
        ws = self.win_size
        stride = max(cs, bs // 2)
        # Contraintes OpenCV
        assert bs % cs == 0, 'block_size doit être multiple de cell_size'
        assert (ws - bs) % stride == 0, 'fenêtre incompatible avec stride'
        return cv2.HOGDescriptor(
            _winSize=(ws, ws),
            _blockSize=(bs, bs),
            _blockStride=(stride, stride),
            _cellSize=(cs, cs),
            _nbins=self.nbins,
        )

    def _extract_one(self, img, hog):
        gray    = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        resized = cv2.resize(gray, (self.win_size, self.win_size))
        feat    = hog.compute(resized).flatten().astype(np.float32)
        norm    = np.linalg.norm(feat)
        return feat / norm if norm > 0 else feat

    def _extract_all(self, X):
        hog  = self._build_hog()
        feats = [self._extract_one(img, hog) for img in X]
        # Padding si dimensions différentes (ne devrait pas arriver)
        max_d = max(f.shape[0] for f in feats)
        out   = np.zeros((len(feats), max_d), dtype=np.float32)
        for i, f in enumerate(feats):
            out[i, :f.shape[0]] = f
        return out

    def fit(self, X, y):
        self.db_features_ = self._extract_all(X)
        self.db_labels_   = np.array(y)
        return self

    def score(self, X, y):
        query_feats  = self._extract_all(X)
        query_labels = np.array(y)
        # Recherche dans la base d'entraînement
        db = l2_normalize(self.db_features_.astype(np.float32))
        qf = l2_normalize(query_feats.astype(np.float32))
        sim = qf @ db.T    # (n_queries, n_db)

        aps = []
        for i in range(len(qf)):
            ranked   = np.argsort(-sim[i])
            relevant = (self.db_labels_[ranked] == query_labels[i]).astype(int)
            n_rel    = relevant.sum()
            if n_rel == 0:
                continue
            k        = min(50, len(ranked))
            rel_k    = relevant[:k]
            prec_k   = np.cumsum(rel_k) / (np.arange(1, k + 1))
            aps.append((prec_k * rel_k).sum() / n_rel)
        return float(np.mean(aps)) if aps else 0.0


print('HOGRetriever défini.')

In [ ]:
# ─── SIFT Estimator ───────────────────────────────────────────────────────

class SIFTRetriever(BaseEstimator):
    """
    Paramètres :
      nfeatures          : nombre max de keypoints
      nOctaveLayers      : couches par octave
      contrastThreshold  : seuil de rejet bas contraste
      edgeThreshold      : seuil de rejet bords
      sigma              : sigma gaussien initial
    Représentation : mean pooling des descripteurs → 128 dim L2-normalisé
    """
    def __init__(self, nfeatures=500, nOctaveLayers=3,
                 contrastThreshold=0.04, edgeThreshold=10, sigma=1.6):
        self.nfeatures         = nfeatures
        self.nOctaveLayers     = nOctaveLayers
        self.contrastThreshold = contrastThreshold
        self.edgeThreshold     = edgeThreshold
        self.sigma             = sigma

    def _extract_one(self, img, sift):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, descs = sift.detectAndCompute(gray, None)
        if descs is None or len(descs) == 0:
            return np.zeros(128, dtype=np.float32)
        feat = descs.mean(axis=0).astype(np.float32)
        norm = np.linalg.norm(feat)
        return feat / norm if norm > 0 else feat

    def _extract_all(self, X):
        sift = cv2.SIFT_create(
            nfeatures=self.nfeatures,
            nOctaveLayers=self.nOctaveLayers,
            contrastThreshold=self.contrastThreshold,
            edgeThreshold=self.edgeThreshold,
            sigma=self.sigma,
        )
        return np.array([self._extract_one(img, sift) for img in X], dtype=np.float32)

    def fit(self, X, y):
        self.db_features_ = self._extract_all(X)
        self.db_labels_   = np.array(y)
        return self

    def score(self, X, y):
        query_feats  = self._extract_all(X)
        query_labels = np.array(y)
        db = l2_normalize(self.db_features_.astype(np.float32))
        qf = l2_normalize(query_feats.astype(np.float32))
        sim = qf @ db.T

        aps = []
        for i in range(len(qf)):
            ranked   = np.argsort(-sim[i])
            relevant = (self.db_labels_[ranked] == query_labels[i]).astype(int)
            n_rel    = relevant.sum()
            if n_rel == 0:
                continue
            k      = min(50, len(ranked))
            rel_k  = relevant[:k]
            prec_k = np.cumsum(rel_k) / (np.arange(1, k + 1))
            aps.append((prec_k * rel_k).sum() / n_rel)
        return float(np.mean(aps)) if aps else 0.0


print('SIFTRetriever défini.')

In [ ]:
# ─── ORB Estimator ────────────────────────────────────────────────────────

class ORBRetriever(BaseEstimator):
    """
    Paramètres :
      nfeatures   : nombre max de keypoints
      scaleFactor : facteur d'échelle entre octaves
      nlevels     : nombre de niveaux de la pyramide
      edgeThreshold : taille du bord ignoré
      patchSize   : taille du patch pour BRIEF
      WTA_K       : points comparés pour un bit BRIEF (2, 3 ou 4)
    Représentation : mean pooling float 32 dim, L2-normalisé
    """
    def __init__(self, nfeatures=500, scaleFactor=1.2, nlevels=8,
                 edgeThreshold=31, patchSize=31, WTA_K=2):
        self.nfeatures     = nfeatures
        self.scaleFactor   = scaleFactor
        self.nlevels       = nlevels
        self.edgeThreshold = edgeThreshold
        self.patchSize     = patchSize
        self.WTA_K         = WTA_K

    def _extract_one(self, img, orb):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, descs = orb.detectAndCompute(gray, None)
        if descs is None or len(descs) == 0:
            return np.zeros(32, dtype=np.float32)
        feat = descs.astype(np.float32).mean(axis=0)
        norm = np.linalg.norm(feat)
        return feat / norm if norm > 0 else feat

    def _extract_all(self, X):
        orb = cv2.ORB_create(
            nfeatures=self.nfeatures,
            scaleFactor=self.scaleFactor,
            nlevels=self.nlevels,
            edgeThreshold=self.edgeThreshold,
            patchSize=self.patchSize,
            WTA_K=self.WTA_K,
        )
        return np.array([self._extract_one(img, orb) for img in X], dtype=np.float32)

    def fit(self, X, y):
        self.db_features_ = self._extract_all(X)
        self.db_labels_   = np.array(y)
        return self

    def score(self, X, y):
        query_feats  = self._extract_all(X)
        query_labels = np.array(y)
        db = l2_normalize(self.db_features_.astype(np.float32))
        qf = l2_normalize(query_feats.astype(np.float32))
        sim = qf @ db.T

        aps = []
        for i in range(len(qf)):
            ranked   = np.argsort(-sim[i])
            relevant = (self.db_labels_[ranked] == query_labels[i]).astype(int)
            n_rel    = relevant.sum()
            if n_rel == 0:
                continue
            k      = min(50, len(ranked))
            rel_k  = relevant[:k]
            prec_k = np.cumsum(rel_k) / (np.arange(1, k + 1))
            aps.append((prec_k * rel_k).sum() / n_rel)
        return float(np.mean(aps)) if aps else 0.0


print('ORBRetriever défini.')

## 4. Grid Search — HOG

| Paramètre | Valeurs testées |
|---|---|
| `win_size` | 64, 96, 128 |
| `block_size` | 8, 16 |
| `cell_size` | 4, 8 |
| `nbins` | 6, 9, 12 |

Contrainte : `block_size` doit être multiple de `cell_size` et `(win_size - block_size)` doit être divisible par `block_size // 2`.

In [ ]:
hog_param_grid = {
    'win_size'   : [64, 96, 128],
    'block_size' : [8, 16],
    'cell_size'  : [4, 8],
    'nbins'      : [6, 9, 12],
}

# Filtre les combinaisons invalides (block_size doit être multiple de cell_size)
from sklearn.model_selection import ParameterGrid

def valid_hog_params(p):
    bs, cs, ws = p['block_size'], p['cell_size'], p['win_size']
    stride = max(cs, bs // 2)
    return (bs % cs == 0) and ((ws - bs) % stride == 0)

all_hog_params = [p for p in ParameterGrid(hog_param_grid) if valid_hog_params(p)]
print(f'{len(all_hog_params)} combinaisons valides sur {len(list(ParameterGrid(hog_param_grid)))} totales')

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

hog_gs = GridSearchCV(
    estimator  = HOGRetriever(),
    param_grid = hog_param_grid,
    cv         = cv,
    n_jobs     = 1,      # cv2 n'est pas thread-safe, éviter n_jobs > 1
    verbose    = 2,
    refit      = True,
)

t0 = time.time()
hog_gs.fit(images_ram, y)
print(f'\nDurée HOG grid search : {time.time() - t0:.1f} s')
print(f'Meilleurs paramètres : {hog_gs.best_params_}')
print(f'Meilleure MAP        : {hog_gs.best_score_:.4f}')

In [ ]:
# Tableau des résultats HOG
hog_results = pd.DataFrame(hog_gs.cv_results_)
hog_results = hog_results.sort_values('mean_test_score', ascending=False)

cols = ['param_win_size', 'param_block_size', 'param_cell_size',
        'param_nbins', 'mean_test_score', 'std_test_score']
hog_results[cols].head(10).rename(columns={
    'mean_test_score': 'MAP_mean',
    'std_test_score' : 'MAP_std'
}).round(4)

In [ ]:
# Heatmap nbins vs block_size (win_size et cell_size fixés aux valeurs du meilleur)
best_ws = hog_gs.best_params_['win_size']
best_cs = hog_gs.best_params_['cell_size']

pivot_data = hog_results[
    (hog_results['param_win_size'] == best_ws) &
    (hog_results['param_cell_size'] == best_cs)
].pivot_table(
    index='param_block_size',
    columns='param_nbins',
    values='mean_test_score'
)

fig, ax = plt.subplots(figsize=(6, 3))
sns.heatmap(pivot_data, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax)
ax.set_title(f'MAP HOG — win_size={best_ws}, cell_size={best_cs}')
ax.set_xlabel('nbins')
ax.set_ylabel('block_size')
plt.tight_layout()
plt.show()

## 5. Grid Search — SIFT

| Paramètre | Valeurs testées |
|---|---|
| `nfeatures` | 100, 300, 500 |
| `nOctaveLayers` | 3, 5 |
| `contrastThreshold` | 0.03, 0.04, 0.06 |
| `edgeThreshold` | 5, 10 |
| `sigma` | 1.2, 1.6 |

⚠️ SIFT est lent (~57 ms/image). Avec 230 images × 3 folds × 72 combinaisons, prévoir ~15-20 min.

In [ ]:
sift_param_grid = {
    'nfeatures'         : [100, 300, 500],
    'nOctaveLayers'     : [3, 5],
    'contrastThreshold' : [0.03, 0.04, 0.06],
    'edgeThreshold'     : [5, 10],
    'sigma'             : [1.2, 1.6],
}

n_sift = len(list(ParameterGrid(sift_param_grid)))
print(f'{n_sift} combinaisons SIFT')
print(f'Temps estimé : {n_sift * 3 * len(images_ram) * 0.057 / 60:.0f} min')

In [ ]:
sift_gs = GridSearchCV(
    estimator  = SIFTRetriever(),
    param_grid = sift_param_grid,
    cv         = cv,
    n_jobs     = 1,
    verbose    = 2,
    refit      = True,
)

t0 = time.time()
sift_gs.fit(images_ram, y)
print(f'\nDurée SIFT grid search : {time.time() - t0:.1f} s')
print(f'Meilleurs paramètres : {sift_gs.best_params_}')
print(f'Meilleure MAP        : {sift_gs.best_score_:.4f}')

In [ ]:
sift_results = pd.DataFrame(sift_gs.cv_results_)
sift_results = sift_results.sort_values('mean_test_score', ascending=False)

cols = ['param_nfeatures', 'param_nOctaveLayers', 'param_contrastThreshold',
        'param_edgeThreshold', 'param_sigma', 'mean_test_score', 'std_test_score']
sift_results[cols].head(10).rename(columns={
    'mean_test_score': 'MAP_mean',
    'std_test_score' : 'MAP_std'
}).round(4)

In [ ]:
# Impact de nfeatures et contrastThreshold (sigma et edgeThreshold fixés aux meilleurs)
best_sigma = sift_gs.best_params_['sigma']
best_edge  = sift_gs.best_params_['edgeThreshold']
best_nl    = sift_gs.best_params_['nOctaveLayers']

pivot_sift = sift_results[
    (sift_results['param_sigma'] == best_sigma) &
    (sift_results['param_edgeThreshold'] == best_edge) &
    (sift_results['param_nOctaveLayers'] == best_nl)
].pivot_table(
    index='param_nfeatures',
    columns='param_contrastThreshold',
    values='mean_test_score'
)

fig, ax = plt.subplots(figsize=(6, 3))
sns.heatmap(pivot_sift, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax)
ax.set_title(f'MAP SIFT — sigma={best_sigma}, edgeThr={best_edge}, nOctave={best_nl}')
ax.set_xlabel('contrastThreshold')
ax.set_ylabel('nfeatures')
plt.tight_layout()
plt.show()

## 6. Grid Search — ORB

| Paramètre | Valeurs testées |
|---|---|
| `nfeatures` | 100, 300, 500, 1000 |
| `scaleFactor` | 1.1, 1.2, 1.5 |
| `nlevels` | 4, 8, 12 |
| `edgeThreshold` | 15, 31 |
| `patchSize` | 15, 31 |
| `WTA_K` | 2, 3, 4 |

In [ ]:
orb_param_grid = {
    'nfeatures'    : [100, 300, 500, 1000],
    'scaleFactor'  : [1.1, 1.2, 1.5],
    'nlevels'      : [4, 8, 12],
    'edgeThreshold': [15, 31],
    'patchSize'    : [15, 31],
    'WTA_K'        : [2, 3, 4],
}

n_orb = len(list(ParameterGrid(orb_param_grid)))
print(f'{n_orb} combinaisons ORB')
print(f'Temps estimé : {n_orb * 3 * len(images_ram) * 0.006 / 60:.0f} min')

In [ ]:
orb_gs = GridSearchCV(
    estimator  = ORBRetriever(),
    param_grid = orb_param_grid,
    cv         = cv,
    n_jobs     = 1,
    verbose    = 2,
    refit      = True,
)

t0 = time.time()
orb_gs.fit(images_ram, y)
print(f'\nDurée ORB grid search : {time.time() - t0:.1f} s')
print(f'Meilleurs paramètres : {orb_gs.best_params_}')
print(f'Meilleure MAP        : {orb_gs.best_score_:.4f}')

In [ ]:
orb_results = pd.DataFrame(orb_gs.cv_results_)
orb_results = orb_results.sort_values('mean_test_score', ascending=False)

cols = ['param_nfeatures', 'param_scaleFactor', 'param_nlevels',
        'param_edgeThreshold', 'param_patchSize', 'param_WTA_K',
        'mean_test_score', 'std_test_score']
orb_results[cols].head(10).rename(columns={
    'mean_test_score': 'MAP_mean',
    'std_test_score' : 'MAP_std'
}).round(4)

In [ ]:
# Impact de nfeatures vs scaleFactor (autres fixés aux meilleurs)
best_nl_orb  = orb_gs.best_params_['nlevels']
best_et_orb  = orb_gs.best_params_['edgeThreshold']
best_ps_orb  = orb_gs.best_params_['patchSize']
best_wta_orb = orb_gs.best_params_['WTA_K']

pivot_orb = orb_results[
    (orb_results['param_nlevels']       == best_nl_orb) &
    (orb_results['param_edgeThreshold'] == best_et_orb) &
    (orb_results['param_patchSize']     == best_ps_orb) &
    (orb_results['param_WTA_K']         == best_wta_orb)
].pivot_table(
    index='param_nfeatures',
    columns='param_scaleFactor',
    values='mean_test_score'
)

fig, ax = plt.subplots(figsize=(6, 3))
sns.heatmap(pivot_orb, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax)
ax.set_title('MAP ORB — nfeatures vs scaleFactor')
ax.set_xlabel('scaleFactor')
ax.set_ylabel('nfeatures')
plt.tight_layout()
plt.show()

## 7. Récapitulatif et comparaison

In [ ]:
summary = pd.DataFrame([
    {'Descripteur': 'HOG',  'MAP_cv': hog_gs.best_score_,  'Meilleurs params': str(hog_gs.best_params_)},
    {'Descripteur': 'SIFT', 'MAP_cv': sift_gs.best_score_, 'Meilleurs params': str(sift_gs.best_params_)},
    {'Descripteur': 'ORB',  'MAP_cv': orb_gs.best_score_,  'Meilleurs params': str(orb_gs.best_params_)},
])
summary['MAP_cv'] = summary['MAP_cv'].round(4)
print(summary.to_string(index=False))

In [ ]:
# Barplot de comparaison
fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(summary['Descripteur'], summary['MAP_cv'],
              color=['steelblue', 'darkorange', 'seagreen'], width=0.4)
ax.bar_label(bars, fmt='%.4f', padding=2)
ax.set_ylim(0, summary['MAP_cv'].max() * 1.2)
ax.set_ylabel('MAP@50 (cross-validation)')
ax.set_title('Meilleure MAP par descripteur après grid search')
plt.tight_layout()
plt.show()

## 8. Validation sur le jeu de données complet

On re-indexe les 5000 images avec les meilleurs hyperparamètres trouvés.

In [ ]:
# Chargement de toutes les images (peut prendre ~2 min)
all_filenames = np.load(os.path.join(INDEX_DIR, 'filenames.npy'))
all_classes   = np.load(os.path.join(INDEX_DIR, 'classes.npy'))
_, y_all = np.unique(all_classes, return_inverse=True)

print(f'Chargement de {len(all_filenames)} images...')
all_images = []
for fname in tqdm(all_filenames, desc='Load full dataset'):
    img = cv2.imread(os.path.join(DATASET_DIR, fname))
    if img is None:
        img = np.zeros((128, 128, 3), dtype=np.uint8)
    all_images.append(img)
all_images = np.array(all_images, dtype=object)
print('Chargement terminé.')

In [ ]:
def full_map(estimator, images, labels):
    """Extrait toutes les features et calcule la MAP globale (leave-one-out)."""
    estimator.fit(images, labels)
    feats = l2_normalize(estimator.db_features_.astype(np.float32))
    return compute_map(feats, labels, top_k=50)


print('Validation HOG sur 5000 images...')
t0 = time.time()
map_hog_full = full_map(HOGRetriever(**hog_gs.best_params_), all_images, y_all)
print(f'  MAP HOG  = {map_hog_full:.4f}  ({time.time()-t0:.1f}s)')

print('Validation SIFT sur 5000 images (peut prendre ~5 min)...')
t0 = time.time()
map_sift_full = full_map(SIFTRetriever(**sift_gs.best_params_), all_images, y_all)
print(f'  MAP SIFT = {map_sift_full:.4f}  ({time.time()-t0:.1f}s)')

print('Validation ORB sur 5000 images...')
t0 = time.time()
map_orb_full = full_map(ORBRetriever(**orb_gs.best_params_), all_images, y_all)
print(f'  MAP ORB  = {map_orb_full:.4f}  ({time.time()-t0:.1f}s)')

In [ ]:
# Comparaison CV vs full dataset
final = pd.DataFrame([
    {'Descripteur': 'HOG',  'MAP_cv (230 img)': hog_gs.best_score_,  'MAP_full (5000 img)': map_hog_full},
    {'Descripteur': 'SIFT', 'MAP_cv (230 img)': sift_gs.best_score_, 'MAP_full (5000 img)': map_sift_full},
    {'Descripteur': 'ORB',  'MAP_cv (230 img)': orb_gs.best_score_,  'MAP_full (5000 img)': map_orb_full},
])
final = final.set_index('Descripteur').round(4)
print(final.to_string())

In [ ]:
# Sauvegarde des meilleurs paramètres
import json

best_params = {
    'hog' : hog_gs.best_params_,
    'sift': sift_gs.best_params_,
    'orb' : orb_gs.best_params_,
}

out_path = os.path.join(PROJECT_ROOT, 'best_descriptor_params.json')
with open(out_path, 'w') as f:
    json.dump(best_params, f, indent=2)

print(f'Meilleurs paramètres sauvegardés dans {out_path}')
print(json.dumps(best_params, indent=2))